API_KEY = "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"

In [ ]:
!pip install groq python-dotenv
import os
os.environ["GROQ_API_KEY"] = "" 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 5.4 MB/s eta 0:00:00


In [11]:
"""
Stage: TRANSCRIPTS_PARSED
Deterministic parser — no LLM calls.
Reads all .txt files from transcripts/ and writes structured JSON to parsed_transcripts/
"""

import os, re, json
from pathlib import Path

TRANSCRIPTS_DIR = Path("/kaggle/input/datasets/deveshswarnkar/data-set")
OUTPUT_DIR      = Path("/kaggle/working/parsed_transcripts")
OUTPUT_DIR.mkdir(exist_ok=True)

HEADER_RE = re.compile(r"---\s*(T-\d+)\s*\(Agent:\s*([^)]+)\)\s*---")

ESCALATION_SIGNALS = [
    r"\bi'm going to report\b",
    r"\bi will report\b",
    r"\bescalate\b",
    r"\bspeak to (a |your )?(manager|supervisor|senior)\b",
    r"\bunacceptable\b",
    r"\bterrible service\b",
    r"\blegal action\b",
    r"\blawyer\b",
]

ISSUE_KEYWORDS = {
    "withdrawal": "Withdrawal delay",
    "payment":    "Payment issue",
    "suspend":    "Account suspension",
    "fraud":      "Fraud flag",
    "leverage":   "Leverage / regulatory query",
    "deposit":    "Deposit query",
    "login":      "Login issue",
    "account":    "Account inquiry",
}

def detect_issue(text: str) -> str:
    low = text.lower()
    for kw, label in ISSUE_KEYWORDS.items():
        if kw in low:
            return label
    return "General inquiry"

def detect_escalation(turns: list) -> list:
    signals = []
    for t in turns:
        if t["speaker"] == "Customer":
            for pattern in ESCALATION_SIGNALS:
                if re.search(pattern, t["text"], re.IGNORECASE):
                    signals.append({"turn": t["turn_number"], "text": t["text"]})
                    break
    return signals

def detect_resolution(turns: list, agent_name: str) -> str:
    agent_turns = [t["text"].lower() for t in turns if t["speaker"] == agent_name]
    combined = " ".join(agent_turns)
    if any(w in combined for w in ["escalat", "senior reviewer", "priority"]):
        return "Escalated"
    if any(w in combined for w in ["resolved", "fixed", "sorted", "done"]):
        return "Resolved"
    if any(w in combined for w in ["have a great day", "have a lovely day", "thanks for calling"]):
        return "Closed — outcome unclear"
    return "Unresolved"

def parse_transcript(filepath: Path) -> dict:
    raw = filepath.read_text(encoding="utf-8").strip()
    lines = [l.strip() for l in raw.splitlines() if l.strip()]

    # Parse header
    header_match = HEADER_RE.match(lines[0])
    if not header_match:
        raise ValueError(f"Cannot parse header in {filepath.name}: {lines[0]}")
    call_id    = header_match.group(1)
    agent_name = header_match.group(2).strip()

    # Parse turns
    turns = []
    turn_number = 1
    full_text_parts = []

    for line in lines[1:]:
        if ":" not in line:
            continue
        speaker, _, text = line.partition(":")
        speaker = speaker.strip()
        text    = text.strip()
        if not text:
            continue
        turns.append({
            "call_id":     call_id,
            "agent_name":  agent_name,
            "turn_number": turn_number,
            "speaker":     speaker,
            "text":        text,
        })
        full_text_parts.append(f"{speaker}: {text}")
        turn_number += 1

    full_text = "\n".join(full_text_parts)

    result = {
        "call_id":             call_id,
        "agent_name":          agent_name,
        "turns":               turns,
        "turn_count":          len(turns),
        "estimated_duration_seconds": len(turns) * 30,
        "issue_type":          detect_issue(full_text),
        "resolution_status":   detect_resolution(turns, agent_name),
        "escalation_signals":  detect_escalation(turns),
        "source_file":         str(filepath),
    }
    return result

def run():
    txt_files = sorted(TRANSCRIPTS_DIR.glob("*.txt"))
    if not txt_files:
        raise FileNotFoundError("No .txt files found in transcripts/")

    all_parsed = []
    for fp in txt_files:
        parsed = parse_transcript(fp)
        out_path = OUTPUT_DIR / f"{parsed['call_id']}.json"
        out_path.write_text(json.dumps(parsed, indent=2), encoding="utf-8")
        print(f"  ✓ {fp.name} → {out_path.name}  "
              f"({parsed['turn_count']} turns | {parsed['issue_type']} | {parsed['resolution_status']})")
        all_parsed.append(parsed)

    print(f"\nTotal parsed: {len(all_parsed)} transcripts")
    return all_parsed

if __name__ == "__main__":
    run()

  ✓ T-1041.txt → T-1041.json  (11 turns | Withdrawal delay | Escalated)
  ✓ T-1042.txt → T-1042.json  (11 turns | Account suspension | Unresolved)
  ✓ T-1043.txt → T-1043.json  (11 turns | Leverage / regulatory query | Closed — outcome unclear)

Total parsed: 3 transcripts


In [13]:
"""
Cell 3 — Stage: QA_SCORED
One separate Groq LLM call per transcript.
Weighted score calculated in Python — never delegated to LLM.
Logs every call to llm_calls.jsonl
"""

import os, json, hashlib, datetime
from pathlib import Path
from groq import Groq

# ── Paths ────────────────────────────────────────────────
INPUT_DIR      = Path("/kaggle/input/datasets/deveshswarnkar/data-set")
FRAMEWORK_PATH = INPUT_DIR / "qa_framework.json"
TRANSCRIPTS_DIR= INPUT_DIR / "transcripts"
PARSED_DIR     = Path("/kaggle/working/parsed_transcripts")
QA_SCORES_PATH = Path("/kaggle/working/qa_scores.json")
LLM_CALLS_PATH = Path("/kaggle/working/llm_calls.jsonl")

# ── Verify paths exist before doing anything ─────────────
print("Verifying paths...")
assert FRAMEWORK_PATH.exists(), f"❌ Not found: {FRAMEWORK_PATH}"
assert PARSED_DIR.exists(),     f"❌ Not found: {PARSED_DIR} — run Cell 2 first"
print(f"  ✅ Framework  : {FRAMEWORK_PATH}")
print(f"  ✅ Parsed dir : {PARSED_DIR}")
print(f"  ✅ Output dir : /kaggle/working/")

# ── Groq client ──────────────────────────────────────────
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
MODEL  = "llama-3.3-70b-versatile"

# ── Load framework ───────────────────────────────────────
framework  = json.loads(FRAMEWORK_PATH.read_text())
dimensions = framework["dimensions"]
auto_fails = framework["auto_fail_conditions"]
weight_map = {d["id"]: d["weight"] for d in dimensions}

print(f"\n  Framework loaded — {len(dimensions)} dimensions, {len(auto_fails)} auto-fail conditions")

# ── Helper: log LLM call ─────────────────────────────────
def log_llm_call(stage, call_id, prompt, input_artifacts, output_artifact, qa_scores_included=False):
    record = {
        "stage":              stage,
        "call_id":            call_id,
        "timestamp":          datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "provider":           "groq",
        "model":              MODEL,
        "prompt_hash":        hashlib.sha256(prompt.encode()).hexdigest(),
        "input_artifacts":    input_artifacts,
        "output_artifact":    output_artifact,
        "qa_scores_included": qa_scores_included,
    }
    with open(LLM_CALLS_PATH, "a") as f:
        f.write(json.dumps(record) + "\n")

# ── Helper: calculate weighted score in Python ───────────
def calculate_weighted_score(dimension_scores: list) -> float:
    total = 0.0
    for ds in dimension_scores:
        total += ds["score"] * weight_map[ds["dimension_id"]]
    return round(total, 4)

# ── Build QA prompt ──────────────────────────────────────
def build_qa_prompt(parsed: dict) -> str:
    transcript_text = "\n".join(
        f"{t['speaker']}: {t['text']}"
        for t in parsed["turns"]
    )
    dims_text = "\n".join(
        f'  - {d["id"]}: {d["name"]} (weight: {d["weight"]})'
        for d in dimensions
    )
    af_text = "\n".join(
        f'  {i+1}. {c}'
        for i, c in enumerate(auto_fails)
    )

    prompt = f"""You are a senior QA analyst evaluating a customer support call transcript.

CALL ID: {parsed['call_id']}
AGENT: {parsed['agent_name']}

=== TRANSCRIPT ===
{transcript_text}

=== QA FRAMEWORK ===
Score each dimension from 0 to 10 (10 = perfect).

DIMENSIONS:
{dims_text}

AUTO-FAIL CONDITIONS (check each explicitly):
{af_text}

=== INSTRUCTIONS ===
Return a single valid JSON object with exactly this structure:

{{
  "call_id": "{parsed['call_id']}",
  "agent_name": "{parsed['agent_name']}",
  "dimension_scores": [
    {{
      "dimension_id": "D1",
      "score": <integer 0-10>,
      "evidence_quote": "<exact quote from transcript>",
      "rationale": "<one sentence>"
    }}
  ],
  "auto_fail_checks": [
    {{
      "condition": "<exact condition text>",
      "triggered": <true|false>,
      "evidence": "<exact quote or null>"
    }}
  ]
}}

Rules:
- dimension_scores must have exactly 7 entries, one per dimension D1 through D7
- auto_fail_checks must have exactly {len(auto_fails)} entries, one per condition listed above
- evidence_quote must be an exact quote from the transcript, not paraphrased
- score must be an integer between 0 and 10
- Return ONLY the JSON object, no explanation, no markdown, no code fences
"""
    return prompt

# ── Main scoring loop ─────────────────────────────────────
def run_qa_scoring():
    parsed_files = sorted(PARSED_DIR.glob("*.json"))
    if not parsed_files:
        raise FileNotFoundError("❌ No parsed transcripts found. Run Cell 2 first.")

    all_results = []

    for fp in parsed_files:
        parsed  = json.loads(fp.read_text())
        call_id = parsed["call_id"]
        agent   = parsed["agent_name"]

        print(f"\n{'─'*55}")
        print(f"  Scoring {call_id} — Agent: {agent}")
        print(f"{'─'*55}")

        prompt = build_qa_prompt(parsed)

        # ── One separate LLM call per transcript ──────────
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            max_tokens=2000,
        )

        raw_output = response.choices[0].message.content.strip()

        # Strip markdown fences if model adds them
        if raw_output.startswith("```"):
            raw_output = raw_output.split("```")[1]
            if raw_output.startswith("json"):
                raw_output = raw_output[4:]
        raw_output = raw_output.strip()

        # Parse JSON
        try:
            result = json.loads(raw_output)
        except json.JSONDecodeError as e:
            print(f"  ❌ JSON parse error: {e}")
            print(f"  Raw output preview: {raw_output[:300]}")
            raise

        # ── Weighted score calculated in Python ───────────
        weighted_score        = calculate_weighted_score(result["dimension_scores"])
        result["weighted_score"]      = weighted_score
        result["auto_fail_triggered"] = any(
            af["triggered"] for af in result["auto_fail_checks"]
        )

        # ── Print summary ─────────────────────────────────
        print(f"  Weighted score : {weighted_score:.2f} / 10")
        print(f"  Auto-fail      : {'⚠️  YES' if result['auto_fail_triggered'] else '✅ No'}")
        print(f"\n  Dimension breakdown:")
        for ds in result["dimension_scores"]:
            bar = "█" * ds["score"] + "░" * (10 - ds["score"])
            print(f"    {ds['dimension_id']}  {bar}  {ds['score']}/10  — {ds['rationale'][:50]}")

        print(f"\n  Auto-fail checks:")
        for af in result["auto_fail_checks"]:
            icon = "⚠️  TRIGGERED" if af["triggered"] else "✅ Clear    "
            print(f"    {icon} — {af['condition'][:50]}")

        # ── Log the LLM call ──────────────────────────────
        log_llm_call(
            stage              = "QA_SCORING",
            call_id            = call_id,
            prompt             = prompt,
            input_artifacts    = [str(fp), str(FRAMEWORK_PATH)],
            output_artifact    = str(QA_SCORES_PATH),
            qa_scores_included = False,
        )

        all_results.append(result)

    # ── Save all results ──────────────────────────────────
    QA_SCORES_PATH.write_text(json.dumps(all_results, indent=2))

    print(f"\n{'='*55}")
    print(f"  ✅ QA scoring complete — {len(all_results)} transcripts scored")
    print(f"  📄 Saved  : {QA_SCORES_PATH}")
    print(f"  📋 Logged : {len(all_results)} entries in llm_calls.jsonl")
    print(f"{'='*55}")

    return all_results

# ── Execute ───────────────────────────────────────────────
results = run_qa_scoring()

Verifying paths...
  ✅ Framework  : /kaggle/input/datasets/deveshswarnkar/data-set/qa_framework.json
  ✅ Parsed dir : /kaggle/working/parsed_transcripts
  ✅ Output dir : /kaggle/working/

  Framework loaded — 7 dimensions, 4 auto-fail conditions

───────────────────────────────────────────────────────
  Scoring T-1041 — Agent: Sara
───────────────────────────────────────────────────────
  Weighted score : 10.00 / 10
  Auto-fail      : ✅ No

  Dimension breakdown:
    D1  ██████████  10/10  — The agent provided a proper greeting and identific
    D2  ██████████  10/10  — The agent demonstrated a clear understanding of th
    D3  ██████████  10/10  — The agent provided accurate information about the 
    D4  ██████████  10/10  — The agent used proper regulatory and compliance la
    D5  ██████████  10/10  — The agent showed empathy and used a proper tone.
    D6  ██████████  10/10  — The agent clearly explained the resolution and nex
    D7  ██████████  10/10  — The agent properly closed

In [14]:
# ============================================================
# CELL 4 — STAGE: COMPLIANCE_EXTRACTED
# One separate Groq LLM call per transcript.
# Focused ONLY on compliance, legal, regulatory, conduct risk.
# Does NOT receive or use QA scores.
# ============================================================

import os, json, hashlib, datetime
from pathlib import Path
from groq import Groq

# ── Paths ────────────────────────────────────────────────
INPUT_DIR       = Path("/kaggle/input/datasets/deveshswarnkar/data-set")
FRAMEWORK_PATH  = INPUT_DIR / "qa_framework.json"
PARSED_DIR      = Path("/kaggle/working/parsed_transcripts")
COMPLIANCE_PATH = Path("/kaggle/working/compliance_flags.json")
LLM_CALLS_PATH  = Path("/kaggle/working/llm_calls.jsonl")

# ── Verify paths ─────────────────────────────────────────
print("Verifying paths...")
assert FRAMEWORK_PATH.exists(), f"❌ Not found: {FRAMEWORK_PATH}"
assert PARSED_DIR.exists(),     f"❌ Not found: {PARSED_DIR} — run Cell 2 first"
print(f"  ✅ Framework  : {FRAMEWORK_PATH}")
print(f"  ✅ Parsed dir : {PARSED_DIR}")

# ── Groq client ──────────────────────────────────────────
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
MODEL  = "llama-3.3-70b-versatile"

# ── Load framework (auto-fail conditions only) ────────────
framework  = json.loads(FRAMEWORK_PATH.read_text())
auto_fails = framework["auto_fail_conditions"]

# ── Helper: log LLM call ─────────────────────────────────
def log_llm_call(stage, call_id, prompt, input_artifacts, output_artifact, qa_scores_included=False):
    record = {
        "stage":              stage,
        "call_id":            call_id,
        "timestamp":          datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "provider":           "groq",
        "model":              MODEL,
        "prompt_hash":        hashlib.sha256(prompt.encode()).hexdigest(),
        "input_artifacts":    input_artifacts,
        "output_artifact":    output_artifact,
        "qa_scores_included": qa_scores_included,
    }
    with open(LLM_CALLS_PATH, "a") as f:
        f.write(json.dumps(record) + "\n")

# ── Build compliance prompt ───────────────────────────────
def build_compliance_prompt(parsed: dict) -> str:
    transcript_text = "\n".join(
        f"{t['speaker']}: {t['text']}"
        for t in parsed["turns"]
    )
    af_text = "\n".join(
        f"  {i+1}. {c}"
        for i, c in enumerate(auto_fails)
    )

    prompt = f"""You are a compliance and risk officer reviewing a customer support call transcript.
Your job is to identify ALL compliance, legal, financial, regulatory, reputational, or conduct risks.

CALL ID: {parsed['call_id']}
AGENT: {parsed['agent_name']}

=== TRANSCRIPT ===
{transcript_text}

=== KNOWN AUTO-FAIL CONDITIONS (flag if any appear) ===
{af_text}

=== RISK TYPES ===
- regulatory          : breach of regulatory rules or guidelines
- financial_commitment: agent makes promises about money, refunds, or timelines
- data_disclosure     : customer data shared without proper identity verification
- conduct             : unprofessional, dismissive, or inappropriate agent behaviour
- reputational        : statements that could damage the company's reputation
- other               : any other risk not covered above

=== SEVERITY LEVELS ===
- critical : immediate escalation required, serious regulatory or legal breach
- high     : significant risk, requires prompt review
- medium   : moderate risk, should be flagged for coaching
- low      : minor concern, worth noting

=== INSTRUCTIONS ===
Identify every genuine risk present in this transcript.
Do not force flags where no risk exists.

Return a single valid JSON object with exactly this structure:

{{
  "call_id": "{parsed['call_id']}",
  "agent_name": "{parsed['agent_name']}",
  "risk_flags": [
    {{
      "call_id": "{parsed['call_id']}",
      "statement_text": "<exact quote from transcript>",
      "speaker": "<speaker name>",
      "risk_type": "<regulatory|financial_commitment|data_disclosure|conduct|reputational|other>",
      "severity": "<critical|high|medium|low>",
      "explanation": "<one or two sentences explaining the risk>"
    }}
  ],
  "total_flags": <integer>,
  "highest_severity": "<critical|high|medium|low|none>"
}}

Rules:
- statement_text must be an exact quote from the transcript
- If there are no risks, return risk_flags as empty list and highest_severity as "none"
- Return ONLY the JSON object, no explanation, no markdown, no code fences
"""
    return prompt

# ── Severity ordering for comparison ─────────────────────
SEVERITY_ORDER = {"critical": 4, "high": 3, "medium": 2, "low": 1, "none": 0}

def highest_severity(flags: list) -> str:
    if not flags:
        return "none"
    return max(flags, key=lambda f: SEVERITY_ORDER.get(f["severity"], 0))["severity"]

# ── Main compliance loop ──────────────────────────────────
def run_compliance_extraction():
    parsed_files = sorted(PARSED_DIR.glob("*.json"))
    if not parsed_files:
        raise FileNotFoundError("❌ No parsed transcripts found. Run Cell 2 first.")

    all_results = []

    for fp in parsed_files:
        parsed  = json.loads(fp.read_text())
        call_id = parsed["call_id"]
        agent   = parsed["agent_name"]

        print(f"\n{'─'*55}")
        print(f"  Compliance check: {call_id} — Agent: {agent}")
        print(f"{'─'*55}")

        prompt = build_compliance_prompt(parsed)

        # ── One separate LLM call per transcript ──────────
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            max_tokens=2000,
        )

        raw_output = response.choices[0].message.content.strip()

        # Strip markdown fences if model adds them
        if raw_output.startswith("```"):
            raw_output = raw_output.split("```")[1]
            if raw_output.startswith("json"):
                raw_output = raw_output[4:]
        raw_output = raw_output.strip()

        # Parse JSON
        try:
            result = json.loads(raw_output)
        except json.JSONDecodeError as e:
            print(f"  ❌ JSON parse error: {e}")
            print(f"  Raw output preview: {raw_output[:300]}")
            raise

        # Recalculate highest severity in Python (don't trust LLM)
        result["highest_severity"] = highest_severity(result.get("risk_flags", []))
        result["total_flags"]      = len(result.get("risk_flags", []))

        # ── Print summary ─────────────────────────────────
        flags = result.get("risk_flags", [])
        print(f"  Total flags      : {result['total_flags']}")
        print(f"  Highest severity : {result['highest_severity'].upper() if flags else 'NONE'}")

        if flags:
            print(f"\n  Risk flags found:")
            for flag in flags:
                sev   = flag['severity'].upper()
                rtype = flag['risk_type']
                quote = flag['statement_text'][:60]
                icon  = {"CRITICAL":"🔴","HIGH":"🟠","MEDIUM":"🟡","LOW":"🟢"}.get(sev, "⚪")
                print(f"    {icon} [{sev}] {rtype}")
                print(f"       \"{quote}...\"")
                print(f"       → {flag['explanation'][:80]}")
        else:
            print("  ✅ No compliance risks identified")

        # ── Log the LLM call ──────────────────────────────
        log_llm_call(
            stage              = "COMPLIANCE_EXTRACTION",
            call_id            = call_id,
            prompt             = prompt,
            input_artifacts    = [str(fp), str(FRAMEWORK_PATH)],
            output_artifact    = str(COMPLIANCE_PATH),
            qa_scores_included = False,
        )

        all_results.append(result)

    # ── Save all compliance flags ─────────────────────────
    COMPLIANCE_PATH.write_text(json.dumps(all_results, indent=2))

    print(f"\n{'='*55}")
    print(f"  ✅ Compliance extraction complete — {len(all_results)} transcripts")
    print(f"  📄 Saved  : {COMPLIANCE_PATH}")
    print(f"  📋 Logged : {len(all_results)} entries in llm_calls.jsonl")
    print(f"{'='*55}")

    return all_results

# ── Execute ───────────────────────────────────────────────
results = run_compliance_extraction()

Verifying paths...
  ✅ Framework  : /kaggle/input/datasets/deveshswarnkar/data-set/qa_framework.json
  ✅ Parsed dir : /kaggle/working/parsed_transcripts

───────────────────────────────────────────────────────
  Compliance check: T-1041 — Agent: Sara
───────────────────────────────────────────────────────
  Total flags      : 2
  Highest severity : MEDIUM

  Risk flags found:
    🟡 [MEDIUM] financial_commitment
       "I can't guarantee same-day processing, but escalating will e..."
       → The agent makes a statement about the timeline for processing the customer's wit
    🟢 [LOW] regulatory
       "It looks like it's been flagged for a manual compliance revi..."
       → The agent discloses the reason for the delay, which is a regulatory requirement,

───────────────────────────────────────────────────────
  Compliance check: T-1042 — Agent: Marcus
───────────────────────────────────────────────────────
  Total flags      : 3
  Highest severity : MEDIUM

  Risk flags found:
    🟡 [M

In [15]:
# ============================================================
# CELL 5 — STAGE: COACHING_GENERATED
# One separate Groq LLM call per transcript.
# STRICTLY — prompt does NOT include:
#   - QA scores
#   - weighted totals
#   - dashboard grades
#   - compliance severity scores
# Only receives: parsed transcript + agent name + call ID
# ============================================================

import os, json, hashlib, datetime
from pathlib import Path
from groq import Groq

# ── Paths ────────────────────────────────────────────────
PARSED_DIR     = Path("/kaggle/working/parsed_transcripts")
COACHING_PATH  = Path("/kaggle/working/coaching_notes.md")
LLM_CALLS_PATH = Path("/kaggle/working/llm_calls.jsonl")

# ── Verify paths ─────────────────────────────────────────
print("Verifying paths...")
assert PARSED_DIR.exists(), f"❌ Not found: {PARSED_DIR} — run Cell 2 first"
print(f"  ✅ Parsed dir : {PARSED_DIR}")
print(f"  ✅ Output     : {COACHING_PATH}")

# ── Groq client ──────────────────────────────────────────
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
MODEL  = "llama-3.3-70b-versatile"

# ── Helper: log LLM call ─────────────────────────────────
def log_llm_call(stage, call_id, prompt, input_artifacts, output_artifact, qa_scores_included=False):
    record = {
        "stage":              stage,
        "call_id":            call_id,
        "timestamp":          datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "provider":           "groq",
        "model":              MODEL,
        "prompt_hash":        hashlib.sha256(prompt.encode()).hexdigest(),
        "input_artifacts":    input_artifacts,
        "output_artifact":    output_artifact,
        "qa_scores_included": qa_scores_included,
    }
    with open(LLM_CALLS_PATH, "a") as f:
        f.write(json.dumps(record) + "\n")

# ── Build coaching prompt ─────────────────────────────────
# IMPORTANT: this function only accepts parsed transcript data.
# QA scores, weighted totals, grades, compliance scores are
# deliberately excluded — enforced by what we pass in.
def build_coaching_prompt(parsed: dict) -> str:

    # ── Safety check: assert no scores are being injected ─
    forbidden_keys = ["weighted_score", "dimension_scores", "auto_fail_checks",
                      "compliance_score", "grade", "dashboard"]
    for key in forbidden_keys:
        assert key not in parsed, f"❌ FORBIDDEN: '{key}' must not be in coaching prompt input"

    transcript_text = "\n".join(
        f"[Turn {t['turn_number']}] {t['speaker']}: {t['text']}"
        for t in parsed["turns"]
    )

    prompt = f"""You are an experienced contact centre coach providing personalised coaching feedback.
Your feedback must be grounded ONLY in evidence from the transcript below.
Do not make assumptions beyond what is explicitly said.

CALL ID   : {parsed['call_id']}
AGENT NAME: {parsed['agent_name']}
ISSUE TYPE: {parsed['issue_type']}
RESOLUTION: {parsed['resolution_status']}

=== TRANSCRIPT ===
{transcript_text}

=== COACHING INSTRUCTIONS ===
Write a structured coaching note for {parsed['agent_name']} based solely on this transcript.

Your coaching note MUST include:

1. WHAT WENT WELL (minimum 2 specific examples)
   - Each point must include the EXACT agent quote from the transcript
   - Explain WHY it was effective

2. DEVELOPMENT AREAS (minimum 2 specific examples)
   - Each point must include the EXACT agent quote from the transcript
   - Explain what the impact was or could be
   - Provide a suggested alternative phrasing the agent could use instead

Rules:
- Every coaching point must reference an exact quote from the transcript
- Do not refer to scores, grades, numbers, or ratings of any kind
- Do not mention QA frameworks, dimensions, or evaluation systems
- Write in a warm, constructive, coaching tone — not punitive
- Be specific — generic feedback without transcript evidence is not acceptable

Format your response in clean markdown with these exact headers:
## Coaching Note — {parsed['agent_name']} ({parsed['call_id']})
### What went well
### Development areas
### Summary
"""
    return prompt

# ── Main coaching loop ────────────────────────────────────
def run_coaching():
    parsed_files = sorted(PARSED_DIR.glob("*.json"))
    if not parsed_files:
        raise FileNotFoundError("❌ No parsed transcripts found. Run Cell 2 first.")

    all_coaching_notes = []

    for fp in parsed_files:
        parsed  = json.loads(fp.read_text())
        call_id = parsed["call_id"]
        agent   = parsed["agent_name"]

        print(f"\n{'─'*55}")
        print(f"  Coaching: {call_id} — Agent: {agent}")
        print(f"{'─'*55}")

        # ── Build prompt with ONLY transcript data ────────
        prompt = build_coaching_prompt(parsed)

        # ── Confirm qa_scores_included = False ────────────
        qa_scores_included = False
        print(f"  qa_scores_included : {qa_scores_included} ✅")

        # ── One LLM call per transcript ───────────────────
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,     # slightly higher for natural coaching tone
            max_tokens=1500,
        )

        coaching_text = response.choices[0].message.content.strip()

        print(f"  ✅ Coaching note generated ({len(coaching_text)} chars)")
        print(f"  Preview:")
        for line in coaching_text.splitlines()[:6]:
            print(f"    {line}")
        print(f"    ...")

        # ── Log the LLM call ──────────────────────────────
        log_llm_call(
            stage              = "COACHING_GENERATION",
            call_id            = call_id,
            prompt             = prompt,
            input_artifacts    = [str(fp)],
            output_artifact    = str(COACHING_PATH),
            qa_scores_included = qa_scores_included,   # always False
        )

        all_coaching_notes.append({
            "call_id":    call_id,
            "agent_name": agent,
            "note":       coaching_text,
        })

    # ── Write all notes to coaching_notes.md ─────────────
    md_lines = ["# Agent Coaching Notes\n"]
    md_lines.append(f"_Generated: {datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}_\n")
    md_lines.append("---\n")

    for item in all_coaching_notes:
        md_lines.append(item["note"])
        md_lines.append("\n\n---\n")

    COACHING_PATH.write_text("\n".join(md_lines), encoding="utf-8")

    print(f"\n{'='*55}")
    print(f"  ✅ Coaching complete — {len(all_coaching_notes)} notes generated")
    print(f"  📄 Saved  : {COACHING_PATH}")
    print(f"  📋 Logged : {len(all_coaching_notes)} entries in llm_calls.jsonl")
    print(f"  🔒 qa_scores_included = False for ALL coaching calls")
    print(f"{'='*55}")

    return all_coaching_notes

# ── Execute ───────────────────────────────────────────────
coaching_results = run_coaching()

Verifying paths...
  ✅ Parsed dir : /kaggle/working/parsed_transcripts
  ✅ Output     : /kaggle/working/coaching_notes.md

───────────────────────────────────────────────────────
  Coaching: T-1041 — Agent: Sara
───────────────────────────────────────────────────────
  qa_scores_included : False ✅
  ✅ Coaching note generated (2675 chars)
  Preview:
    ## Coaching Note — Sara (T-1041)
    ### What went well
    Sara demonstrated excellent customer service skills in several areas. Firstly, when the customer expressed frustration, Sara responded with empathy: "I completely understand your frustration, and I appreciate your patience." This was effective because it acknowledged the customer's feelings and showed that Sara was actively listening. By doing so, Sara helped to de-escalate the situation and create a more positive tone for the rest of the call.
    
    Another example of what went well was when Sara explained the reason for the withdrawal delay: "It looks like it's been flagged

In [16]:
# ============================================================
# CELL 6 — STAGE: DASHBOARD_COMPUTED
# Pure Python — NO LLM call.
# Reads qa_scores.json + compliance_flags.json
# Calculates grades with deterministic thresholds in code.
# Writes dashboard_summary.md
# ============================================================

import json, datetime
from pathlib import Path

# ── Paths ────────────────────────────────────────────────
QA_SCORES_PATH   = Path("/kaggle/working/qa_scores.json")
COMPLIANCE_PATH  = Path("/kaggle/working/compliance_flags.json")
DASHBOARD_PATH   = Path("/kaggle/working/dashboard_summary.md")

# ── Verify inputs exist ───────────────────────────────────
print("Verifying inputs...")
assert QA_SCORES_PATH.exists(),  f"❌ Not found: {QA_SCORES_PATH}  — run Cell 3 first"
assert COMPLIANCE_PATH.exists(), f"❌ Not found: {COMPLIANCE_PATH} — run Cell 4 first"
print(f"  ✅ QA scores      : {QA_SCORES_PATH}")
print(f"  ✅ Compliance     : {COMPLIANCE_PATH}")

# ── Load data ─────────────────────────────────────────────
qa_scores        = json.loads(QA_SCORES_PATH.read_text())
compliance_flags = json.loads(COMPLIANCE_PATH.read_text())

# Build compliance lookup by call_id
compliance_lookup = {c["call_id"]: c for c in compliance_flags}

# ── Grade thresholds — deterministic in code ──────────────
# A >= 85  |  B >= 70  |  C >= 55  |  D >= 40  |  F < 40
# F also if any auto-fail condition is triggered
def calculate_grade(weighted_score: float, auto_fail_triggered: bool) -> str:
    if auto_fail_triggered:
        return "F"
    scaled = weighted_score * 10      # convert 0-10 score to 0-100 scale
    if scaled >= 85:
        return "A"
    elif scaled >= 70:
        return "B"
    elif scaled >= 55:
        return "C"
    elif scaled >= 40:
        return "D"
    else:
        return "F"

GRADE_EMOJI = {"A": "🟢", "B": "🟡", "C": "🟠", "D": "🔴", "F": "🔴"}
SEVERITY_ORDER = {"critical": 4, "high": 3, "medium": 2, "low": 1, "none": 0}

# ── Build dashboard rows ──────────────────────────────────
def build_dashboard():
    rows = []

    for qa in qa_scores:
        call_id           = qa["call_id"]
        agent_name        = qa["agent_name"]
        weighted_score    = qa["weighted_score"]
        auto_fail         = qa["auto_fail_triggered"]

        # Get compliance data for this call
        comp              = compliance_lookup.get(call_id, {})
        flags             = comp.get("risk_flags", [])
        compliance_count  = len(flags)
        highest_severity  = comp.get("highest_severity", "none")

        # ── Grade calculated in Python ────────────────────
        grade = calculate_grade(weighted_score, auto_fail)

        row = {
            "call_id":           call_id,
            "agent_name":        agent_name,
            "weighted_score":    weighted_score,
            "scaled_score":      round(weighted_score * 10, 2),
            "auto_fail":         auto_fail,
            "compliance_count":  compliance_count,
            "highest_severity":  highest_severity,
            "grade":             grade,
        }
        rows.append(row)

        # Print summary per call
        emoji = GRADE_EMOJI.get(grade, "⚪")
        print(f"\n  {emoji} {call_id} — {agent_name}")
        print(f"     Weighted score   : {weighted_score:.4f} → {round(weighted_score*10,2)}/100")
        print(f"     Auto-fail        : {'⚠️  YES → Grade forced to F' if auto_fail else '✅ No'}")
        print(f"     Compliance flags : {compliance_count}  (highest: {highest_severity.upper()})")
        print(f"     Grade            : {grade}")

    return rows

# ── Write dashboard_summary.md ────────────────────────────
def write_dashboard(rows: list):
    now = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

    lines = []
    lines.append("# QA Dashboard Summary")
    lines.append(f"\n_Generated: {now}_\n")
    lines.append("---\n")

    # ── Grade scale reference ─────────────────────────────
    lines.append("## Grade Scale")
    lines.append("| Grade | Score Range | Notes |")
    lines.append("|-------|-------------|-------|")
    lines.append("| A     | ≥ 85        | Excellent |")
    lines.append("| B     | ≥ 70        | Good |")
    lines.append("| C     | ≥ 55        | Satisfactory |")
    lines.append("| D     | ≥ 40        | Needs improvement |")
    lines.append("| F     | < 40 or auto-fail triggered | Fail |\n")

    # ── Main results table ────────────────────────────────
    lines.append("## Results\n")
    lines.append("| Call ID | Agent | Score (/100) | Auto-Fail | Compliance Flags | Highest Severity | Grade |")
    lines.append("|---------|-------|-------------|-----------|-----------------|-----------------|-------|")

    for row in rows:
        af_cell  = "⚠️ YES" if row["auto_fail"] else "✅ No"
        sev_cell = row["highest_severity"].upper()
        grade    = row["grade"]
        emoji    = GRADE_EMOJI.get(grade, "")
        lines.append(
            f"| {row['call_id']} "
            f"| {row['agent_name']} "
            f"| {row['scaled_score']} "
            f"| {af_cell} "
            f"| {row['compliance_count']} "
            f"| {sev_cell} "
            f"| {emoji} **{grade}** |"
        )

    lines.append("")

    # ── Per-agent detail cards ────────────────────────────
    lines.append("## Agent Detail\n")

    for row in rows:
        grade = row["grade"]
        emoji = GRADE_EMOJI.get(grade, "")
        lines.append(f"### {emoji} {row['agent_name']} — {row['call_id']}")
        lines.append(f"- **Weighted score** : {row['weighted_score']:.4f} / 10  →  {row['scaled_score']} / 100")
        lines.append(f"- **Auto-fail triggered** : {'Yes ⚠️' if row['auto_fail'] else 'No ✅'}")
        lines.append(f"- **Compliance flags** : {row['compliance_count']}")
        lines.append(f"- **Highest severity** : {row['highest_severity'].upper()}")
        lines.append(f"- **Overall grade** : **{grade}**")
        lines.append("")

    # ── Team summary ──────────────────────────────────────
    lines.append("## Team Summary\n")
    avg_score   = sum(r["scaled_score"] for r in rows) / len(rows)
    total_flags = sum(r["compliance_count"] for r in rows)
    fails       = sum(1 for r in rows if r["grade"] == "F")
    grade_dist  = {}
    for r in rows:
        grade_dist[r["grade"]] = grade_dist.get(r["grade"], 0) + 1

    lines.append(f"- **Calls reviewed**    : {len(rows)}")
    lines.append(f"- **Average score**     : {avg_score:.2f} / 100")
    lines.append(f"- **Total compliance flags** : {total_flags}")
    lines.append(f"- **Auto-fail / F grades**   : {fails}")
    lines.append(f"- **Grade distribution** : " + " | ".join(f"{g}: {c}" for g, c in sorted(grade_dist.items())))
    lines.append("\n---")
    lines.append("_Grades computed deterministically in Python. No LLM involvement in grading._")

    DASHBOARD_PATH.write_text("\n".join(lines), encoding="utf-8")

# ── Execute ───────────────────────────────────────────────
print(f"\n{'─'*55}")
print("  Building dashboard...")
print(f"{'─'*55}")

rows = build_dashboard()
write_dashboard(rows)

print(f"\n{'='*55}")
print(f"  ✅ Dashboard complete — {len(rows)} agents")
print(f"  📄 Saved : {DASHBOARD_PATH}")
print(f"  🔒 All grades calculated in Python — no LLM")
print(f"{'='*55}")

Verifying inputs...
  ✅ QA scores      : /kaggle/working/qa_scores.json
  ✅ Compliance     : /kaggle/working/compliance_flags.json

───────────────────────────────────────────────────────
  Building dashboard...
───────────────────────────────────────────────────────

  🟢 T-1041 — Sara
     Weighted score   : 10.0000 → 100.0/100
     Auto-fail        : ✅ No
     Compliance flags : 2  (highest: MEDIUM)
     Grade            : A

  🔴 T-1042 — Marcus
     Weighted score   : 7.9000 → 79.0/100
     Auto-fail        : ⚠️  YES → Grade forced to F
     Compliance flags : 3  (highest: MEDIUM)
     Grade            : F

  🟢 T-1043 — Linda
     Weighted score   : 10.0000 → 100.0/100
     Auto-fail        : ✅ No
     Compliance flags : 2  (highest: LOW)
     Grade            : A

  ✅ Dashboard complete — 3 agents
  📄 Saved : /kaggle/working/dashboard_summary.md
  🔒 All grades calculated in Python — no LLM


In [18]:
# ============================================================
# CELL 7 — SHOULD: CALIBRATION CHECK + TEAM TREND
# One LLM call for calibration check (all scoresheets)
# Team trend is pure Python — no LLM
# ============================================================

import os, json, re, hashlib, datetime
from pathlib import Path
from groq import Groq

# ── Paths ────────────────────────────────────────────────
QA_SCORES_PATH   = Path("/kaggle/working/qa_scores.json")
CALIBRATION_PATH = Path("/kaggle/working/calibration_check.json")
TEAM_TREND_PATH  = Path("/kaggle/working/team_trend.json")
LLM_CALLS_PATH   = Path("/kaggle/working/llm_calls.jsonl")

# ── Verify ────────────────────────────────────────────────
print("Verifying inputs...")
assert QA_SCORES_PATH.exists(), f"❌ Not found: {QA_SCORES_PATH} — run Cell 3 first"
print(f"  ✅ QA scores : {QA_SCORES_PATH}")

# ── Groq client ──────────────────────────────────────────
client    = Groq(api_key=os.environ.get("GROQ_API_KEY"))
MODEL     = "llama-3.3-70b-versatile"
qa_scores = json.loads(QA_SCORES_PATH.read_text())

# ── Helper: robust JSON extraction ───────────────────────
def extract_json(raw: str) -> dict:
    """
    Tries multiple strategies to extract valid JSON from LLM output.
    Handles markdown fences, truncated text, and trailing garbage.
    """
    # Strategy 1: strip markdown fences
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        parts   = cleaned.split("```")
        cleaned = parts[1] if len(parts) > 1 else cleaned
        if cleaned.startswith("json"):
            cleaned = cleaned[4:]
        cleaned = cleaned.strip()

    # Strategy 2: direct parse
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # Strategy 3: find outermost { ... } block
    start = cleaned.find("{")
    end   = cleaned.rfind("}")
    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(cleaned[start:end+1])
        except json.JSONDecodeError:
            pass

    # Strategy 4: truncate at last complete array element
    # Find last valid closing bracket before a potential cut-off
    for end_char in ["]}", "}}"]:
        idx = cleaned.rfind(end_char)
        if idx != -1:
            candidate = cleaned[:idx+2]
            # Try to close any open structures
            try:
                return json.loads(candidate)
            except json.JSONDecodeError:
                pass

    raise json.JSONDecodeError("Could not extract valid JSON", cleaned, 0)

# ── Helper: log LLM call ─────────────────────────────────
def log_llm_call(stage, call_id, prompt, input_artifacts, output_artifact, qa_scores_included=False):
    record = {
        "stage":              stage,
        "call_id":            call_id,
        "timestamp":          datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "provider":           "groq",
        "model":              MODEL,
        "prompt_hash":        hashlib.sha256(prompt.encode()).hexdigest(),
        "input_artifacts":    input_artifacts,
        "output_artifact":    output_artifact,
        "qa_scores_included": qa_scores_included,
    }
    with open(LLM_CALLS_PATH, "a") as f:
        f.write(json.dumps(record) + "\n")

# ============================================================
# PART A — CALIBRATION CHECK (one LLM call, all scoresheets)
# ============================================================

def build_calibration_prompt(qa_scores: list) -> str:
    scoresheets = ""
    for qa in qa_scores:
        scoresheets += f"\n--- {qa['call_id']} | Agent: {qa['agent_name']} ---\n"
        for ds in qa.get("dimension_scores", []):
            scoresheets += (
                f"  {ds['dimension_id']}: score={ds['score']}  "
                f"rationale=\"{ds['rationale'][:80]}\"\n"
            )

    prompt = f"""You are a QA calibration specialist reviewing scoring consistency.

Below are dimension scores for {len(qa_scores)} evaluated calls.
Identify where similar agent behaviours received meaningfully different scores.

=== ALL SCORESHEETS ===
{scoresheets}

=== INSTRUCTIONS ===
Return ONLY a valid JSON object. No explanation, no markdown, no code fences.
Keep all string values SHORT — under 100 characters each.

{{
  "inconsistencies": [
    {{
      "dimension_id": "D1",
      "call_ids": ["T-1041", "T-1042"],
      "observed_difference": "short description under 80 chars",
      "why_it_may_be_inconsistent": "short reason under 80 chars",
      "recommended_adjustment": "short suggestion under 80 chars"
    }}
  ],
  "overall_calibration_comment": "one short paragraph under 200 chars"
}}

If no inconsistencies exist, return an empty inconsistencies array.
IMPORTANT: Keep every string value concise. Return ONLY the JSON object.
"""
    return prompt

def run_calibration(max_retries: int = 3) -> dict:
    print(f"\n{'─'*55}")
    print("  Running calibration check (1 LLM call — all scoresheets)")
    print(f"{'─'*55}")

    prompt = build_calibration_prompt(qa_scores)

    for attempt in range(1, max_retries + 1):
        print(f"  Attempt {attempt}/{max_retries}...")

        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            max_tokens=1500,
        )

        raw_output = response.choices[0].message.content.strip()

        try:
            result = extract_json(raw_output)
            print(f"  ✅ JSON parsed successfully on attempt {attempt}")
            break
        except json.JSONDecodeError as e:
            print(f"  ⚠️  Attempt {attempt} failed: {e}")
            print(f"  Raw output preview:\n{raw_output[:400]}\n")
            if attempt == max_retries:
                # Final fallback: return a safe default structure
                print("  ⚠️  All attempts failed — saving fallback structure")
                result = {
                    "inconsistencies": [],
                    "overall_calibration_comment": (
                        "Calibration check could not be parsed from LLM output. "
                        "Manual review recommended."
                    ),
                    "parse_error": str(e),
                    "raw_output":  raw_output[:500],
                }

    inconsistencies = result.get("inconsistencies", [])
    print(f"  Inconsistencies found : {len(inconsistencies)}")
    for inc in inconsistencies:
        print(f"    {inc['dimension_id']} | {inc['call_ids']} | {inc['observed_difference'][:60]}")
    print(f"  Comment: {result.get('overall_calibration_comment','')[:100]}...")

    log_llm_call(
        stage              = "CALIBRATION_CHECK",
        call_id            = "ALL",
        prompt             = prompt,
        input_artifacts    = [str(QA_SCORES_PATH)],
        output_artifact    = str(CALIBRATION_PATH),
        qa_scores_included = True,
    )

    CALIBRATION_PATH.write_text(json.dumps(result, indent=2))
    print(f"  ✅ Saved : {CALIBRATION_PATH}")
    return result

# ============================================================
# PART B — TEAM TREND (pure Python — no LLM)
# ============================================================

HISTORICAL_BASELINE = {
    "D1": 8, "D2": 7, "D3": 7,
    "D4": 6, "D5": 7, "D6": 7, "D7": 8
}

def run_team_trend() -> dict:
    print(f"\n{'─'*55}")
    print("  Computing team trend (pure Python — no LLM)")
    print(f"{'─'*55}")

    # Calculate current average per dimension
    dim_totals = {}
    dim_counts = {}

    for qa in qa_scores:
        for ds in qa.get("dimension_scores", []):
            did = ds["dimension_id"]
            dim_totals[did] = dim_totals.get(did, 0) + ds["score"]
            dim_counts[did] = dim_counts.get(did, 0) + 1

    current_averages = {
        did: round(dim_totals[did] / dim_counts[did], 2)
        for did in dim_totals
    }

    trend_rows = []
    print(f"\n  {'DIM':<5} {'BASELINE':>10} {'CURRENT':>10} {'DELTA':>8}  STATUS")
    print(f"  {'─'*50}")

    for did in sorted(HISTORICAL_BASELINE.keys()):
        baseline = HISTORICAL_BASELINE[did]
        current  = current_averages.get(did, 0)
        delta    = round(current - baseline, 2)
        status   = "▲ Improved" if delta > 0 else ("▼ Declined" if delta < 0 else "► Same")
        print(f"  {did:<5} {baseline:>10} {current:>10} {delta:>+8.2f}  {status}")
        trend_rows.append({
            "dimension_id":    did,
            "baseline_score":  baseline,
            "current_average": current,
            "delta":           delta,
            "status":          status,
        })

    overall_current  = round(sum(current_averages.values()) / len(current_averages), 2)
    overall_baseline = round(sum(HISTORICAL_BASELINE.values()) / len(HISTORICAL_BASELINE), 2)
    overall_delta    = round(overall_current - overall_baseline, 2)

    result = {
        "generated_at":       datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "historical_baseline": HISTORICAL_BASELINE,
        "current_averages":   current_averages,
        "trend_by_dimension": trend_rows,
        "overall_baseline":   overall_baseline,
        "overall_current":    overall_current,
        "overall_delta":      overall_delta,
        "overall_status":     "Improved" if overall_delta > 0 else ("Declined" if overall_delta < 0 else "Same"),
        "calls_included":     [qa["call_id"] for qa in qa_scores],
    }

    print(f"\n  {'─'*50}")
    print(f"  Overall baseline : {overall_baseline}")
    print(f"  Overall current  : {overall_current}")
    print(f"  Overall delta    : {overall_delta:+.2f} → {result['overall_status']}")

    TEAM_TREND_PATH.write_text(json.dumps(result, indent=2))
    print(f"\n  ✅ Saved : {TEAM_TREND_PATH}")
    return result

# ── Execute ───────────────────────────────────────────────
calibration_result = run_calibration()
trend_result       = run_team_trend()

print(f"\n{'='*55}")
print(f"  ✅ Cell 7 complete")
print(f"  📄 calibration_check.json saved")
print(f"  📄 team_trend.json saved")
print(f"{'='*55}")

Verifying inputs...
  ✅ QA scores : /kaggle/working/qa_scores.json

───────────────────────────────────────────────────────
  Running calibration check (1 LLM call — all scoresheets)
───────────────────────────────────────────────────────
  Attempt 1/3...
  ✅ JSON parsed successfully on attempt 1
  Inconsistencies found : 2
    D5 | ['T-1041', 'T-1042'] | empathy score
    D2 | ['T-1041', 'T-1042'] | issue understanding
  Comment: inconsistent scoring...
  ✅ Saved : /kaggle/working/calibration_check.json

───────────────────────────────────────────────────────
  Computing team trend (pure Python — no LLM)
───────────────────────────────────────────────────────

  DIM     BASELINE    CURRENT    DELTA  STATUS
  ──────────────────────────────────────────────────
  D1             8       10.0    +2.00  ▲ Improved
  D2             7       9.33    +2.33  ▲ Improved
  D3             7       9.67    +2.67  ▲ Improved
  D4             6       9.33    +3.33  ▲ Improved
  D5             7       8

In [19]:
# ============================================================
# CELL 8 — STRETCH: ESCALATION CASES + REBUTTAL COACHING
# Escalation logic is pure Python (no LLM)
# Rebuttal coaching is one LLM call per qualifying transcript
# ============================================================

import os, json, hashlib, datetime
from pathlib import Path
from groq import Groq

# ── Paths ────────────────────────────────────────────────
QA_SCORES_PATH    = Path("/kaggle/working/qa_scores.json")
COMPLIANCE_PATH   = Path("/kaggle/working/compliance_flags.json")
PARSED_DIR        = Path("/kaggle/working/parsed_transcripts")
ESCALATION_PATH   = Path("/kaggle/working/escalation_cases.json")
REBUTTAL_PATH     = Path("/kaggle/working/rebuttal_coaching.md")
LLM_CALLS_PATH    = Path("/kaggle/working/llm_calls.jsonl")

# ── Verify ────────────────────────────────────────────────
print("Verifying inputs...")
assert QA_SCORES_PATH.exists(),  f"❌ Not found: {QA_SCORES_PATH}  — run Cell 3 first"
assert COMPLIANCE_PATH.exists(), f"❌ Not found: {COMPLIANCE_PATH} — run Cell 4 first"
assert PARSED_DIR.exists(),      f"❌ Not found: {PARSED_DIR}      — run Cell 2 first"
print(f"  ✅ QA scores  : {QA_SCORES_PATH}")
print(f"  ✅ Compliance : {COMPLIANCE_PATH}")
print(f"  ✅ Parsed dir : {PARSED_DIR}")

# ── Load data ─────────────────────────────────────────────
client     = Groq(api_key=os.environ.get("GROQ_API_KEY"))
MODEL      = "llama-3.3-70b-versatile"
qa_scores  = json.loads(QA_SCORES_PATH.read_text())
compliance = json.loads(COMPLIANCE_PATH.read_text())

qa_lookup   = {q["call_id"]: q for q in qa_scores}
comp_lookup = {c["call_id"]: c for c in compliance}

# ── Helper: log LLM call ─────────────────────────────────
def log_llm_call(stage, call_id, prompt, input_artifacts, output_artifact, qa_scores_included=False):
    record = {
        "stage":              stage,
        "call_id":            call_id,
        "timestamp":          datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "provider":           "groq",
        "model":              MODEL,
        "prompt_hash":        hashlib.sha256(prompt.encode()).hexdigest(),
        "input_artifacts":    input_artifacts,
        "output_artifact":    output_artifact,
        "qa_scores_included": qa_scores_included,
    }
    with open(LLM_CALLS_PATH, "a") as f:
        f.write(json.dumps(record) + "\n")

# ============================================================
# PART A — ESCALATION CASES (pure Python — no LLM)
# Triggers: auto-fail OR critical compliance flag
# ============================================================

def build_escalation_cases():
    print(f"\n{'─'*55}")
    print("  Building escalation cases (pure Python — no LLM)")
    print(f"{'─'*55}")

    cases = []

    for parsed_file in sorted(PARSED_DIR.glob("*.json")):
        parsed  = json.loads(parsed_file.read_text())
        call_id = parsed["call_id"]
        agent   = parsed["agent_name"]
        qa      = qa_lookup.get(call_id, {})
        comp    = comp_lookup.get(call_id, {})

        auto_fail   = qa.get("auto_fail_triggered", False)
        flags       = comp.get("risk_flags", [])
        has_critical = any(f["severity"] == "critical" for f in flags)

        # ── Check escalation triggers ─────────────────────
        if auto_fail:
            # Find which auto-fail condition was triggered
            triggered_conditions = [
                af for af in qa.get("auto_fail_checks", [])
                if af["triggered"]
            ]
            for tc in triggered_conditions:
                cases.append({
                    "call_id":            call_id,
                    "agent_name":         agent,
                    "trigger_type":       "auto_fail",
                    "triggered_condition": tc["condition"],
                    "transcript_excerpt": tc.get("evidence") or "See full transcript",
                    "recommended_action": (
                        "Immediate supervisor review required. "
                        "Agent must complete refresher training before next shift. "
                        "Document incident in personnel file."
                    ),
                })
            print(f"  ⚠️  {call_id} — AUTO-FAIL escalation ({len(triggered_conditions)} condition(s))")

        if has_critical:
            critical_flags = [f for f in flags if f["severity"] == "critical"]
            for cf in critical_flags:
                cases.append({
                    "call_id":            call_id,
                    "agent_name":         agent,
                    "trigger_type":       "critical_compliance",
                    "triggered_condition": cf["risk_type"],
                    "transcript_excerpt": cf["statement_text"],
                    "recommended_action": (
                        f"Critical {cf['risk_type']} risk identified. "
                        "Escalate to compliance team immediately. "
                        f"Review: {cf['explanation']}"
                    ),
                })
            print(f"  🔴 {call_id} — CRITICAL COMPLIANCE escalation ({len(critical_flags)} flag(s))")

        if not auto_fail and not has_critical:
            print(f"  ✅ {call_id} — No escalation required")

    ESCALATION_PATH.write_text(json.dumps(cases, indent=2))
    print(f"\n  Total escalation cases : {len(cases)}")
    print(f"  ✅ Saved : {ESCALATION_PATH}")
    return cases

# ============================================================
# PART B — REBUTTAL COACHING (LLM call per qualifying call)
# Triggers: customer threatens to report, complain, escalate
# ============================================================

REBUTTAL_TRIGGERS = [
    "going to report",
    "will report",
    "complain",
    "escalate",
    "speak to a manager",
    "speak to your manager",
    "legal action",
    "lawyer",
    "sue",
    "terrible service",
    "unacceptable",
]

def has_rebuttal_trigger(parsed: dict) -> tuple:
    """Returns (True, triggering_text) if customer threatened/complained"""
    for turn in parsed["turns"]:
        if turn["speaker"] == "Customer":
            text_lower = turn["text"].lower()
            for trigger in REBUTTAL_TRIGGERS:
                if trigger in text_lower:
                    return True, turn["text"]
    return False, None

def build_rebuttal_prompt(parsed: dict, trigger_text: str) -> str:
    transcript_text = "\n".join(
        f"[Turn {t['turn_number']}] {t['speaker']}: {t['text']}"
        for t in parsed["turns"]
    )

    prompt = f"""You are an expert contact centre coach specialising in complaint handling and de-escalation.

A customer became hostile or threatening during this call. Your job is to generate a role-play coaching scenario
to help the agent practise handling similar situations better in future.

CALL ID   : {parsed['call_id']}
AGENT NAME: {parsed['agent_name']}
TRIGGERING CUSTOMER STATEMENT: "{trigger_text}"

=== FULL TRANSCRIPT ===
{transcript_text}

=== INSTRUCTIONS ===
Generate a rebuttal coaching scenario in clean markdown with exactly these sections:

## Rebuttal Coaching — {parsed['agent_name']} ({parsed['call_id']})

### Situation summary
(2-3 sentences describing what happened and why it escalated)

### Simulated customer message
(The hostile/threatening message the agent needs to practise responding to)

### Ideal agent response
(A model response the agent should aim for — professional, empathetic, de-escalating)

### Coaching goal
(One sentence: what skill this scenario develops)

### Technique demonstrated
(Name the technique used — e.g. "Acknowledge, Align, Assure" — and explain it in 2-3 sentences)

### Alternative phrases to practise
(3 bullet points of specific phrases the agent can use in similar situations)

Rules:
- Reference exact quotes from the transcript
- Keep tone coaching-focused and constructive
- Do not mention scores, grades, or QA frameworks
"""
    return prompt

def run_rebuttal_coaching():
    print(f"\n{'─'*55}")
    print("  Running rebuttal coaching (LLM call per qualifying call)")
    print(f"{'─'*55}")

    rebuttal_notes = []

    for parsed_file in sorted(PARSED_DIR.glob("*.json")):
        parsed  = json.loads(parsed_file.read_text())
        call_id = parsed["call_id"]
        agent   = parsed["agent_name"]

        triggered, trigger_text = has_rebuttal_trigger(parsed)

        if not triggered:
            print(f"  ⏭️  {call_id} — No rebuttal trigger found, skipping")
            continue

        print(f"\n  🎯 {call_id} — Rebuttal trigger found")
        print(f"     Trigger: \"{trigger_text[:70]}\"")

        prompt = build_rebuttal_prompt(parsed, trigger_text)

        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.4,
            max_tokens=1500,
        )

        rebuttal_text = response.choices[0].message.content.strip()
        print(f"  ✅ Rebuttal coaching generated ({len(rebuttal_text)} chars)")

        log_llm_call(
            stage              = "REBUTTAL_COACHING",
            call_id            = call_id,
            prompt             = prompt,
            input_artifacts    = [str(parsed_file)],
            output_artifact    = str(REBUTTAL_PATH),
            qa_scores_included = False,
        )

        rebuttal_notes.append({
            "call_id":    call_id,
            "agent_name": agent,
            "note":       rebuttal_text,
        })

    # ── Write rebuttal_coaching.md ────────────────────────
    now = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
    md_lines = [
        "# Rebuttal Coaching Scenarios",
        f"\n_Generated: {now}_\n",
        f"_Qualifying calls: {len(rebuttal_notes)}_\n",
        "---\n",
    ]

    if not rebuttal_notes:
        md_lines.append("_No rebuttal coaching scenarios generated — no qualifying triggers found._")
    else:
        for item in rebuttal_notes:
            md_lines.append(item["note"])
            md_lines.append("\n\n---\n")

    REBUTTAL_PATH.write_text("\n".join(md_lines), encoding="utf-8")
    print(f"\n  ✅ Saved : {REBUTTAL_PATH}")
    return rebuttal_notes

# ── Execute ───────────────────────────────────────────────
escalation_cases  = build_escalation_cases()
rebuttal_results  = run_rebuttal_coaching()

print(f"\n{'='*55}")
print(f"  ✅ Cell 8 complete")
print(f"  📄 escalation_cases.json  — {len(escalation_cases)} case(s)")
print(f"  📄 rebuttal_coaching.md   — {len(rebuttal_results)} scenario(s)")
print(f"{'='*55}")

Verifying inputs...
  ✅ QA scores  : /kaggle/working/qa_scores.json
  ✅ Compliance : /kaggle/working/compliance_flags.json
  ✅ Parsed dir : /kaggle/working/parsed_transcripts

───────────────────────────────────────────────────────
  Building escalation cases (pure Python — no LLM)
───────────────────────────────────────────────────────
  ✅ T-1041 — No escalation required
  ⚠️  T-1042 — AUTO-FAIL escalation (1 condition(s))
  ✅ T-1043 — No escalation required

  Total escalation cases : 1
  ✅ Saved : /kaggle/working/escalation_cases.json

───────────────────────────────────────────────────────
  Running rebuttal coaching (LLM call per qualifying call)
───────────────────────────────────────────────────────
  ⏭️  T-1041 — No rebuttal trigger found, skipping

  🎯 T-1042 — Rebuttal trigger found
     Trigger: "Weeks?! That's unacceptable. I have money in there."
  ✅ Rebuttal coaching generated (1934 chars)
  ⏭️  T-1043 — No rebuttal trigger found, skipping

  ✅ Saved : /kaggle/working/reb

In [20]:
# ============================================================
# CELL 9 — VALIDATION
# Checks all required artifacts, stage separation,
# JSON validity, grade logic, and coaching safety.
# Run this last to confirm the full pipeline is correct.
# ============================================================

import json, sys
from pathlib import Path

WORKING = Path("/kaggle/working")
INPUT   = Path("/kaggle/input/datasets/deveshswarnkar/data-set")

PASS = 0
FAIL = 0

def check(label, condition, detail=""):
    global PASS, FAIL
    if condition:
        print(f"  ✅ {label}")
        PASS += 1
    else:
        print(f"  ❌ {label}{' — ' + detail if detail else ''}")
        FAIL += 1

def section(title):
    print(f"\n{'─'*55}")
    print(f"  {title}")
    print(f"{'─'*55}")

print("=" * 55)
print("  PIPELINE VALIDATION")
print("=" * 55)

# ── 1. Required artifacts exist ───────────────────────────
section("1. Required artifacts")
required_files = [
    INPUT  / "qa_framework.json",
    WORKING / "parsed_transcripts" / "T-1041.json",
    WORKING / "parsed_transcripts" / "T-1042.json",
    WORKING / "parsed_transcripts" / "T-1043.json",
    WORKING / "qa_scores.json",
    WORKING / "compliance_flags.json",
    WORKING / "coaching_notes.md",
    WORKING / "dashboard_summary.md",
    WORKING / "llm_calls.jsonl",
]
for f in required_files:
    check(f.name, f.exists(), str(f))

# ── 2. Stretch artifacts ──────────────────────────────────
section("2. Stretch artifacts")
stretch_files = [
    WORKING / "calibration_check.json",
    WORKING / "team_trend.json",
    WORKING / "escalation_cases.json",
    WORKING / "rebuttal_coaching.md",
]
for f in stretch_files:
    check(f.name, f.exists(), str(f))

# ── 3. JSON files are valid ───────────────────────────────
section("3. JSON validity")
json_files = [
    INPUT  / "qa_framework.json",
    WORKING / "qa_scores.json",
    WORKING / "compliance_flags.json",
    WORKING / "calibration_check.json",
    WORKING / "team_trend.json",
    WORKING / "escalation_cases.json",
]
for f in json_files:
    if f.exists():
        try:
            json.loads(f.read_text())
            check(f"{f.name} valid JSON", True)
        except Exception as e:
            check(f"{f.name} valid JSON", False, str(e))

# ── 4. All transcripts were parsed ───────────────────────
section("4. Transcript parsing")
txt_files    = list(INPUT.glob("*.txt"))
parsed_files = list((WORKING / "parsed_transcripts").glob("*.json"))
check("At least one transcript found", len(txt_files) > 0, f"Found: {len(txt_files)}")
check("Parsed count matches transcript count", len(parsed_files) == len(txt_files),
      f"txt={len(txt_files)} parsed={len(parsed_files)}")

for pf in parsed_files:
    p = json.loads(pf.read_text())
    check(f"{pf.name} has required fields",
          all(k in p for k in ["call_id","agent_name","turns","turn_count",
                                "estimated_duration_seconds","issue_type",
                                "resolution_status","escalation_signals"]))
    check(f"{pf.name} turns have required fields",
          all(all(k in t for k in ["call_id","agent_name","turn_number","speaker","text"])
              for t in p["turns"]))

# ── 5. QA dimensions match framework ─────────────────────
section("5. QA scoring correctness")
framework  = json.loads((INPUT / "qa_framework.json").read_text())
dim_ids    = {d["id"] for d in framework["dimensions"]}
weight_map = {d["id"]: d["weight"] for d in framework["dimensions"]}
auto_fails = framework["auto_fail_conditions"]

if (WORKING / "qa_scores.json").exists():
    qa_scores = json.loads((WORKING / "qa_scores.json").read_text())
    for qa in qa_scores:
        cid = qa["call_id"]
        scored_dims = {ds["dimension_id"] for ds in qa.get("dimension_scores", [])}
        check(f"{cid} has all 7 dimensions", scored_dims == dim_ids,
              f"Missing: {dim_ids - scored_dims}")
        check(f"{cid} has all {len(auto_fails)} auto-fail checks",
              len(qa.get("auto_fail_checks",[])) == len(auto_fails))
        check(f"{cid} scores are integers 0-10",
              all(isinstance(ds["score"], int) and 0 <= ds["score"] <= 10
                  for ds in qa.get("dimension_scores",[])))

        # Verify weighted score matches Python calculation
        calc = round(sum(
            ds["score"] * weight_map[ds["dimension_id"]]
            for ds in qa.get("dimension_scores", [])
        ), 4)
        check(f"{cid} weighted score correct",
              abs(qa.get("weighted_score", -1) - calc) < 0.001,
              f"stored={qa.get('weighted_score')} calculated={calc}")

# ── 6. Grades are correct ─────────────────────────────────
section("6. Dashboard grade correctness")
def expected_grade(score, auto_fail):
    if auto_fail: return "F"
    s = score * 10
    if s >= 85: return "A"
    elif s >= 70: return "B"
    elif s >= 55: return "C"
    elif s >= 40: return "D"
    else: return "F"

if (WORKING / "dashboard_summary.md").exists():
    dash_text = (WORKING / "dashboard_summary.md").read_text()
    check("dashboard_summary.md exists and non-empty", len(dash_text) > 100)
    if (WORKING / "qa_scores.json").exists():
        qa_scores = json.loads((WORKING / "qa_scores.json").read_text())
        for qa in qa_scores:
            eg = expected_grade(qa["weighted_score"], qa["auto_fail_triggered"])
            check(f"{qa['call_id']} grade {eg} appears in dashboard",
                  f"**{eg}**" in dash_text or f"| {eg} |" in dash_text or f"Grade {eg}" in dash_text)

# ── 7. LLM call log stage separation ─────────────────────
section("7. LLM call log — stage separation")
if (WORKING / "llm_calls.jsonl").exists():
    records = [json.loads(l) for l in (WORKING / "llm_calls.jsonl").read_text().splitlines() if l.strip()]

    parsed_files_list = list((WORKING / "parsed_transcripts").glob("*.json"))
    n = len(parsed_files_list)

    qa_records         = [r for r in records if r["stage"] == "QA_SCORING"]
    compliance_records = [r for r in records if r["stage"] == "COMPLIANCE_EXTRACTION"]
    coaching_records   = [r for r in records if r["stage"] == "COACHING_GENERATION"]
    calibration_records= [r for r in records if r["stage"] == "CALIBRATION_CHECK"]

    check(f"Separate QA record per transcript ({n} expected)",
          len(qa_records) == n, f"found {len(qa_records)}")
    check(f"Separate compliance record per transcript ({n} expected)",
          len(compliance_records) == n, f"found {len(compliance_records)}")
    check(f"Separate coaching record per transcript ({n} expected)",
          len(coaching_records) == n, f"found {len(coaching_records)}")
    check("Calibration check record exists",
          len(calibration_records) >= 1)

    # Critical: coaching calls must have qa_scores_included = False
    coaching_clean = all(not r["qa_scores_included"] for r in coaching_records)
    check("All coaching calls have qa_scores_included=False", coaching_clean)

    # All records have required fields
    required_log_fields = ["stage","call_id","timestamp","provider","model",
                           "prompt_hash","input_artifacts","output_artifact","qa_scores_included"]
    all_valid = all(all(k in r for k in required_log_fields) for r in records)
    check("All log records have required fields", all_valid)

# ── 8. Coaching notes quote agent turns ──────────────────
section("8. Coaching note quality")
if (WORKING / "coaching_notes.md").exists():
    coaching_text = (WORKING / "coaching_notes.md").read_text()
    check("coaching_notes.md non-empty", len(coaching_text) > 200)
    check("Contains 'What went well' section",    "What went well"    in coaching_text)
    check("Contains 'Development areas' section", "Development areas" in coaching_text)

    # Check at least one exact agent quote appears
    if (WORKING / "parsed_transcripts" / "T-1041.json").exists():
        sara_turns = json.loads((WORKING / "parsed_transcripts" / "T-1041.json").read_text())
        agent_quotes = [t["text"][:30] for t in sara_turns["turns"] if t["speaker"] == "Sara"]
        quote_found  = any(q in coaching_text for q in agent_quotes)
        check("Coaching notes contain exact agent quotes", quote_found)

# ── 9. Compliance flags structure ────────────────────────
section("9. Compliance flags structure")
if (WORKING / "compliance_flags.json").exists():
    comp_data = json.loads((WORKING / "compliance_flags.json").read_text())
    for c in comp_data:
        cid = c["call_id"]
        check(f"{cid} compliance has required fields",
              all(k in c for k in ["call_id","agent_name","risk_flags","total_flags","highest_severity"]))
        valid_types = {"regulatory","financial_commitment","data_disclosure","conduct","reputational","other"}
        valid_sevs  = {"critical","high","medium","low"}
        for flag in c.get("risk_flags",[]):
            check(f"{cid} flag risk_type valid",  flag.get("risk_type")  in valid_types)
            check(f"{cid} flag severity valid",   flag.get("severity")   in valid_sevs)
            check(f"{cid} flag has statement",    bool(flag.get("statement_text")))

# ── Final summary ─────────────────────────────────────────
total = PASS + FAIL
print(f"\n{'='*55}")
print(f"  VALIDATION COMPLETE")
print(f"  Passed : {PASS}/{total}")
print(f"  Failed : {FAIL}/{total}")
if FAIL == 0:
    print(f"  🎉 ALL CHECKS PASSED — pipeline is valid")
else:
    print(f"  ⚠️  {FAIL} check(s) failed — review output above")
print(f"{'='*55}")

if FAIL > 0:
    sys.exit(1)

  PIPELINE VALIDATION

───────────────────────────────────────────────────────
  1. Required artifacts
───────────────────────────────────────────────────────
  ✅ qa_framework.json
  ✅ T-1041.json
  ✅ T-1042.json
  ✅ T-1043.json
  ✅ qa_scores.json
  ✅ compliance_flags.json
  ✅ coaching_notes.md
  ✅ dashboard_summary.md
  ✅ llm_calls.jsonl

───────────────────────────────────────────────────────
  2. Stretch artifacts
───────────────────────────────────────────────────────
  ✅ calibration_check.json
  ✅ team_trend.json
  ✅ escalation_cases.json
  ✅ rebuttal_coaching.md

───────────────────────────────────────────────────────
  3. JSON validity
───────────────────────────────────────────────────────
  ✅ qa_framework.json valid JSON
  ✅ qa_scores.json valid JSON
  ✅ compliance_flags.json valid JSON
  ✅ calibration_check.json valid JSON
  ✅ team_trend.json valid JSON
  ✅ escalation_cases.json valid JSON

───────────────────────────────────────────────────────
  4. Transcript parsing
──────